# <font color=firebrick>Econometrics — M2 MBFA</font>
## TD 3 / 6 — Probit and Logit

**Prerequisites:** Session 4 (CM) — the latent-variable model, the standard normal and logistic CDFs, maximum likelihood, probit and logit.

**Format:** four exercises (Part 1), a guided analysis on a dataset (Part 2), an analysis on a dataset of your choice (Part 3).

**Today's objectives:**
- Get comfortable with the standard normal and logistic distributions, and with the idea of an iid sample.
- Compute a maximum likelihood estimate by hand in a simple case, and see the effect of the logarithm.
- Estimate, test, and interpret a probit/logit model on real data - including marginal effects.

---


## Part 1 — Distributions and likelihood

### 1.1 The standard normal and logistic CDFs

**Task 1.1 - Part A (by hand).** The logistic CDF has a closed form, $\Lambda(z)=\dfrac{e^z}{1+e^z}$, so you can compute it with just a calculator.

1. Compute $\Lambda(0.5)$ by hand (keep 3 decimals).
2. The standard normal CDF $\Phi(z)$ has **no closed form** - it is only available via a table or software. Using the table from Session 4 (or any standard normal table), read off $\Phi(0.5)$.
3. Which of the two, $\Lambda(0.5)$ or $\Phi(0.5)$, is larger? Is this consistent with what Session 4 said about the logistic distribution's spread?

<details>
<summary>Show answer</summary>

$\Lambda(0.5)=\dfrac{e^{0.5}}{1+e^{0.5}}=\dfrac{1.6487}{2.6487}\approx0.623$.

$\Phi(0.5)\approx0.691$ (from a standard normal table).

$\Phi(0.5)>\Lambda(0.5)$: for the same positive $z$, $\Phi$ is already closer to $1$ than $\Lambda$ is - consistent with the logistic distribution having more spread, so it needs a larger $z$ to reach the same probability.

</details>

**Task 1.1 - Part B (computational).** Complete the code below to compute both CDFs at $z=0.5$ with Python, and check they match your hand computation.


In [ ]:
import numpy as np
from scipy.stats import norm

z = 0.5

# TODO: compute Phi(z) with scipy, and Lambda(z) with the closed-form formula
Phi_z = ...
Lambda_z = ...

print("Phi(0.5)   =", Phi_z)
print("Lambda(0.5) =", Lambda_z)
assert np.isclose(Phi_z, 0.6915, atol=1e-3), "Phi(0.5) should be about 0.6915 - check you used scipy's norm.cdf."
assert np.isclose(Lambda_z, 0.6225, atol=1e-3), "Lambda(0.5) should be about 0.6225 - check the closed-form formula."
print("Self-check passed!")


Phi(0.5)   = 0.6914624612740131
Lambda(0.5) = 0.6224593312018546
Self-check passed!


### 1.2 Independence: from individual to joint probability

**Task 1.2 - Part A (by hand).** Three independent policyholders have claim probabilities $p_1=0.2$, $p_2=0.5$, $p_3=0.1$ (they differ because they have different characteristics $x_i$). Suppose we observe $y=(1,0,0)$: the first files a claim, the other two do not.

1. Write down $P(y_1=1)$, $P(y_2=0)$, $P(y_3=0)$.
2. Using independence, compute $P(y_1=1,\,y_2=0,\,y_3=0)$ - this is $L(\cdot)$, the likelihood of this particular sample.

<details>
<summary>Show answer</summary>

$P(y_1=1)=p_1=0.2$. $P(y_2=0)=1-p_2=0.5$. $P(y_3=0)=1-p_3=0.9$.

By independence, $L=0.2\times0.5\times0.9=0.09$.

</details>

**Task 1.2 - Part B (computational).** Complete the code below to compute the same likelihood, then use it to check that a *different* observed sample, $y=(0,1,1)$, would have a different (here, much smaller) likelihood under the same $p_1,p_2,p_3$ - the likelihood depends on what was actually observed, not just on the $p_i$'s.


In [ ]:
p = np.array([0.2, 0.5, 0.1])

y_a = np.array([1, 0, 0])
y_b = np.array([0, 1, 1])

def likelihood(y, p):
    # TODO: return the likelihood of observing `y`, given individual probabilities `p`
    # (hint: p**y * (1-p)**(1-y) gives, for each i, P(y_i=1) if y_i=1 and P(y_i=0) if y_i=0)
    return ...

L_a = likelihood(y_a, p)
L_b = likelihood(y_b, p)
print("L(y=(1,0,0)) =", L_a)
print("L(y=(0,1,1)) =", L_b)
assert np.isclose(L_a, 0.09, atol=1e-6), "L(y=(1,0,0)) should be 0.09 - check your formula."
print("Self-check passed!")


### 1.3 Maximum likelihood, by hand: a new coin-flip example

**Task 1.3 - Part A (by hand).** Five independent policyholders: $y=(0,1,1,1,0)$ - three claims out of five. We want the maximum likelihood estimate $\hat p$ of a *constant* claim probability $p$ (no covariates, exactly as in Session 4's worked example, but with different data).

1. Write $L(p)$ as a function of $p$.
2. Compute $L(p)$ for $p=0.2$, $p=0.4$, $p=0.6$.
3. Which of the three is largest? Does it match $k/n$ (the observed proportion of claims)?

<details>
<summary>Show answer</summary>

$L(p)=(1-p)\times p\times p\times p\times(1-p)=p^3(1-p)^2$.

$L(0.2)=0.008\times0.64=0.00512$. $L(0.4)=0.064\times0.36=0.02304$. $L(0.6)=0.216\times0.16=0.03456$.

$L(0.6)$ is clearly the largest of the three - and $0.6=3/5=k/n$, the observed proportion of claims, exactly as the calculus predicts: differentiating $\log L(p)=3\log p+2\log(1-p)$ and setting it to zero gives $3/p=2/(1-p)$, i.e. $3(1-p)=2p$, i.e. $p=3/5=0.6$.

</details>

**Task 1.3 - Part B (computational).** Complete the code below to verify $L(p)$ at a finer grid of candidate values, and confirm the maximum is at $p=0.6$.


In [ ]:
y = np.array([0, 1, 1, 1, 0])
k = y.sum()
n = len(y)
print(f"{k} claims out of {n}")

p_grid = np.linspace(0.01, 0.99, 99)

# TODO: compute L(p) = p^k * (1-p)^(n-k) for every value in p_grid
L_grid = ...

p_hat = p_grid[np.argmax(L_grid)]
print("p that maximizes L(p) on this grid:", round(p_hat, 2))
assert np.isclose(p_hat, k / n, atol=0.02), "The maximizer should match k/n - check your formula for L_grid."
print("Self-check passed!")


### 1.4 Why the logarithm doesn't change the answer

**Task 1.4 (computational).** Using the same `p_grid` and `L_grid` from Task 1.3, compute `logL_grid = np.log(L_grid)` and find the value of $p$ that maximizes it. Confirm it is the *same* $p$ as the one that maximized $L(p)$ directly - exactly the point Session 4 made: taking the log changes the *shape* of the curve (and makes it numerically manageable), but never *where* its maximum is.


In [ ]:
# TODO: compute the log-likelihood, and find its maximizer
logL_grid = ...
p_hat_log = p_grid[np.argmax(logL_grid)]

print("p that maximizes log L(p):", round(p_hat_log, 2))
assert np.isclose(p_hat_log, p_hat, atol=1e-9), "The maximizer of log(L) should be exactly the same as the maximizer of L."
print("Self-check passed!")


## Part 2 — Guided analysis: mortgage loan approval

We investigate a classic question in consumer finance: what drives whether a mortgage application is approved? We use `loanapp` from `wooldridge` - $1\,989$ real mortgage applications, with `approve` ($=1$ if approved) as the outcome.


In [ ]:
from wooldridge import load_data
import pandas as pd
import statsmodels.formula.api as smf

loanapp = load_data.data('loanapp')
loanapp[['approve', 'obrat', 'married', 'appinc']].describe()


In [ ]:
load_data.data('loanapp', description=True)


### 2.1 First look

**Task 2.1.** What share of applications are approved (`approve.mean()`)? Given this, would a naive rule "always predict approved" already get a fairly high share of predictions right? Why is that not the same as having a *useful* model?


In [ ]:
print("Share approved:", loanapp['approve'].mean())


### 2.2 Estimating probit and logit

We focus on `obrat` (other obligations, as a % of total income - a standard measure of how indebted the applicant already is). The model:
$$
P(\text{approve}_i=1\mid \text{obrat}_i) = F(\beta_0+\beta_1\,\text{obrat}_i).
$$

**Task 2.2.** Estimate both a probit and a logit version below (reuse the code pattern from Session 4). Compare the sign of $\hat\beta_1$ to your economic intuition: should a higher debt ratio make approval more or less likely?


In [ ]:
res_probit = smf.probit('approve ~ obrat', data=loanapp).fit()
print(res_probit.summary())


In [ ]:
res_logit = smf.logit('approve ~ obrat', data=loanapp).fit()
print(res_logit.summary())


### 2.3 Testing and marginal effects

**Task 2.3.**
1. State $H_0$ and $H_1$ for the coefficient on `obrat`. Using the probit table, is it statistically significant at the 5% level?
2. Compute the average marginal effect of `obrat` for both models (as in Session 4). Are they close to each other, even though the raw coefficients differ in scale?


In [ ]:
margeff_probit = res_probit.get_margeff(at='overall')
print(margeff_probit.summary())

margeff_logit = res_logit.get_margeff(at='overall')
print(margeff_logit.summary())


### 2.4 Economic interpretation

**Task 2.4.** Using the probit average marginal effect, translate the effect of `obrat` into a concrete statement: "each additional percentage point of debt-to-income ratio changes the approval probability by about ___ percentage points." Is this economically large or small, given that `obrat` ranges from about $0$ to $95$ in this sample (`loanapp['obrat'].describe()`)?


### 2.5 Adding a second regressor

**Task 2.5 (bonus).** Add `married` to the specification (`approve ~ obrat + married`) and re-estimate. Does the coefficient - or the marginal effect - on `obrat` change much? Relate this to Session 3's omitted variable bias result: does it look like `married` was a relevant, correlated omitted variable, or not?


## Part 3 — Your own analysis

Same principle as TD 2: pick **one** dataset below, and repeat the same kind of analysis as Part 2, making your own decisions about the model - a binary outcome, explained by 2-3 regressors of your choice.

| Dataset | Topic | Size | A binary outcome |
|---|---|---|---|
| `apple` | Eco-friendly product purchase | 660 obs | `ecobuy` - **you must create it**: `ecobuy = (apple['ecolbs'] > 0).astype(int)` |
| `mroz` | Labor force participation | 753 obs | `inlf` (already $0/1$) |
| `smoke` | Smoking decision | 807 obs | `smoker` - **you must create it**: `smoker = (smoke['cigs'] > 0).astype(int)` |
| `fertil2` | Contraceptive use | 4,361 obs | `usemeth` (already $0/1$) |

Load the description of your chosen dataset (`load_data.data('name', description=True)`) before deciding on your model.

### The canvas

Same six steps as Part 2 (and as TD 2's Part 3) - a markdown cell for your written answer, an empty code cell to fill in.

**Step 1 - Load, and build your binary outcome if needed.**


**Step 2 - Choose your regressors and your model.** Write it down; justify each variable economically.

**Step 3 - First look.** What share of the sample has outcome $=1$? Look at the distribution of your explanatory variables.

**Step 4 - Estimate both a probit and a logit.**

**Step 5 - Test.** For your main variable of interest: state $H_0$/$H_1$, and is it significant at 5%?

**Step 6 - Interpret.** Compute the average marginal effect (for either model) of your main variable of interest, and translate it into a concrete, economically meaningful sentence (as in Task 2.4).